# AFTERAI — Colab quickstart

Use this notebook from an Android browser or PC. Choose a free GPU runtime only when one is offered; do not switch to a paid plan. The first model candidate is Qwen/Qwen3-0.6B. This notebook does not start training unless you explicitly set `RUN_TRAINING = True`.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo_dir = Path('/content/AFTERAI')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/huzeifaumar34/AFTERAI.git', str(repo_dir)], check=True)
%cd /content/AFTERAI

In [ ]:
%pip install -e '.[training]'

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU assigned. You can validate data, but model training may be impractically slow on CPU.')

## Upload your reviewed dataset

Upload a `.jsonl` file that you have permission to use. Do not upload private conversations, passwords, API keys, or personal records. The tiny example in the repository only checks the format; it is not a useful training dataset.

In [ ]:
from google.colab import files

uploaded = files.upload()
jsonl_files = [name for name in uploaded if name.lower().endswith('.jsonl')]
if len(jsonl_files) != 1:
    raise ValueError('Upload exactly one .jsonl dataset file.')
dataset_path = str(Path('/content') / jsonl_files[0])
print('Selected dataset:', dataset_path)

In [ ]:
subprocess.run([sys.executable, 'scripts/validate_dataset.py', dataset_path], check=True)

## Optional LoRA training

Training can consume the free runtime allowance and may be interrupted. Keep `RUN_TRAINING` false until you have reviewed the dataset, model license, and available GPU memory.

In [ ]:
RUN_TRAINING = False  # Change to True only when you are ready to start training.

if RUN_TRAINING:
    command = [
        sys.executable, 'scripts/train_lora.py',
        '--model', 'Qwen/Qwen3-0.6B',
        '--dataset', dataset_path,
        '--output', '/content/afterai-lora-output',
        '--epochs', '1',
    ]
    subprocess.run(command, check=True)
else:
    print('Training skipped. Set RUN_TRAINING = True after reviewing the prerequisites.')

In [ ]:
DOWNLOAD_ADAPTER = False  # Change to True after a successful training run.

if RUN_TRAINING and DOWNLOAD_ADAPTER:
    import shutil
    archive_path = shutil.make_archive('/content/afterai-lora-output', 'zip', '/content/afterai-lora-output')
    files.download(archive_path)
else:
    print('No adapter downloaded.')